# Yu-Gi-Oh! Dataset Scrapping

This notebook will be used to scrap Yu-Gi-Oh! Card data. The data will be analyzed and being uesd to get some insights

Data Mining Week 1 - Scrapping & EDA Technique

> Source: https://ygoprodeck.com

In [2]:
import time
import requests
import pandas as pd

API_URL = "https://db.ygoprodeck.com/api/v7/cardinfo.php"

def scrape_cards(batch_size=100):
    cards = []
    offset = 0

    while True:
        params = {
            "num": batch_size,
            "offset": offset
        }

        response = requests.get(API_URL, params=params, timeout=30)

        # No more results or invalid offset
        if response.status_code == 400:
            break

        response.raise_for_status()
        payload = response.json()

        batch = payload.get("data", [])
        if not batch:
            break

        cards.extend(batch)
        print(f"Collected {len(cards)} cards")

        if not payload.get("meta", {}).get("next_page"):
            break

        offset += batch_size

        # Polite delay
        time.sleep(0.1)

    return cards


cards = scrape_cards()

Collected 100 cards
Collected 200 cards
Collected 300 cards
Collected 400 cards
Collected 500 cards
Collected 600 cards
Collected 700 cards
Collected 800 cards
Collected 900 cards
Collected 1000 cards
Collected 1100 cards
Collected 1200 cards
Collected 1300 cards
Collected 1400 cards
Collected 1500 cards
Collected 1600 cards
Collected 1700 cards
Collected 1800 cards
Collected 1900 cards
Collected 2000 cards
Collected 2100 cards
Collected 2200 cards
Collected 2300 cards
Collected 2400 cards
Collected 2500 cards
Collected 2600 cards
Collected 2700 cards
Collected 2800 cards
Collected 2900 cards
Collected 3000 cards
Collected 3100 cards
Collected 3200 cards
Collected 3300 cards
Collected 3400 cards
Collected 3500 cards
Collected 3600 cards
Collected 3700 cards
Collected 3800 cards
Collected 3900 cards
Collected 4000 cards
Collected 4100 cards
Collected 4200 cards
Collected 4300 cards
Collected 4400 cards
Collected 4500 cards
Collected 4600 cards
Collected 4700 cards
Collected 4800 cards
C

In [4]:
def flatten_card(card):
    images = card.get("card_images") or []
    prices = card.get("card_prices") or []
    sets = card.get("card_sets") or []
    banlist = card.get("banlist_info") or {}

    first_image = images[0] if images else {}
    first_price = prices[0] if prices else {}

    return {
        "id": card.get("id"),
        "name": card.get("name"),
        "type": card.get("type"),
        "frame_type": card.get("frameType"),
        "description": card.get("desc"),
        "race": card.get("race"),
        "attribute": card.get("attribute"),
        "archetype": card.get("archetype"),
        "level": card.get("level"),
        "rank": card.get("rank"),
        "link_value": card.get("linkval"),
        "atk": card.get("atk"),
        "def": card.get("def"),

        "image_url": first_image.get("image_url"),
        "image_url_small": first_image.get("image_url_small"),

        "tcgplayer_price": first_price.get("tcgplayer_price"),
        "cardmarket_price": first_price.get("cardmarket_price"),
        "ebay_price": first_price.get("ebay_price"),
        "amazon_price": first_price.get("amazon_price"),

        "ban_tcg": banlist.get("ban_tcg"),
        "ban_ocg": banlist.get("ban_ocg"),
        "ban_goat": banlist.get("ban_goat"),

        "set_count": len(sets),
        "sets": sets, 
    }


df = pd.DataFrame(flatten_card(card) for card in cards)
df.to_csv("data/yugioh_cards.csv", index=False)

print(df.shape)
print(df.head())


(14533, 24)
         id                           name        type frame_type  \
0  80181649                "A Case for K9"  Spell Card      spell   
1  34541863       "A" Cell Breeding Device  Spell Card      spell   
2  64163367             "A" Cell Incubator  Spell Card      spell   
3  91231901  "A" Cell Recombination Device  Spell Card      spell   
4  73262676         "A" Cell Scatter Burst  Spell Card      spell   

                                         description        race attribute  \
0  When this card is activated: You can add 1 "K9...  Continuous      None   
1  During each of your Standby Phases, put 1 A-Co...  Continuous      None   
2  Each time an A-Counter(s) is removed from play...  Continuous      None   
3  Target 1 face-up monster on the field; send 1 ...  Quick-Play      None   
4  Select 1 face-up "Alien" monster you control. ...  Quick-Play      None   

  archetype  level  rank  ...  \
0        K9    NaN  None  ...   
1     Alien    NaN  None  ...   
2    